In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from vocal_assistant.vocal_assistant import VocalAssistant
from vocal_assistant.emotion.emotion_model import process_func, EmotionModel
from transformers import Wav2Vec2Processor
from vocal_assistant.emotion.predict_emotion import load_trained_model, predict_emotion
import pandas as pd


In [ ]:
songs = pd.read_pickle("./data/Songs")


In [ ]:
vc = VocalAssistant(1)

In [ ]:
device = 'cpu'

audeering_model_name = 'audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim'
audeering_processor = Wav2Vec2Processor.from_pretrained(audeering_model_name)
audeering_model = EmotionModel.from_pretrained(audeering_model_name).to(device)

custom_model_name = "model_checkpoint_sampled.pth"
pretrained_model = "facebook/wav2vec2-base"
custom_model, custom_processor = load_trained_model(custom_model_name, pretrained_model)

In [ ]:
vc.talk("What is your mood today?")
"""while True:
    command, vocal_file = vc.take_command()
    print(command)
    break
""" 
file_path = "sad.wav" 
vocal_file = vc.process_audio_file(file_path)
print("Audeering: ")
audeering = process_func(vocal_file, 16000)[0]
print(audeering)
print("Custom: ")
#custom = list(predict_emotion(custom_model, custom_processor, vocal_file).values())
custom = predict_emotion(custom_model, custom_processor, vocal_file)[0].tolist()
print(custom)
#custom model seems to give the same results: overfitting?
#[0.4807744026184082, 0.5821987390518188, 0.6608558893203735]


In [ ]:
import numpy as np
dim_vec = np.array(audeering[0:2])
songs_list = pd.DataFrame({"id": songs["musicId"], "eucl_dist":songs[["Valence", "Arousal"]]\
                           .apply(lambda x: np.linalg.norm(x - dim_vec), axis=1), "Valence": songs["Valence"], "Arousal": songs["Arousal"],\
                            "title":songs["title"], "artist": songs["artist"], "mp3_file":songs["mp3_file"]})

songs_list = songs_list.sort_values(by="eucl_dist")[:5]
songs_list
#create temp dir?

In [ ]:
import sounddevice as sd

for i in range(len(songs_list)):
    sd.play(songs_list["mp3_file"].iloc[i], 44100)
    sd.wait()
